<div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 40px; margin-top: 0;">
    <div style="flex: 0 0 auto; margin-left: 0; margin-bottom: 0; margin-top: 0;">
        <img src="./pics/UCSD Logo.png" alt="UCSD Logo" style="width: 179px; margin-bottom: 0px; margin-top: 20px;">
    </div>
    <div style="flex: 0 0 auto; margin-left: auto; margin-bottom: 0; margin-top: 20px;">
        <img src="./pics/LANL-logo.png" alt="LANL Logo" style="width: 200px; margin-bottom: 0px;">
    </div>
    <div style="flex: 0 0 auto; margin-left: auto; margin-bottom: 0; margin-top: 20px;">
        <img src="./pics/prowess.png" alt="Prowess Logo" style="width: 200px; margin-bottom: 0px;">
    </div>
    <div style="flex: 0 0 auto; margin-left: auto; margin-bottom: 0; margin-top: 20px;">
        <img src="./pics/wildfire.png" alt="WildFire Logo" width="100"/>
    </div>
</div>

<h1 style="text-align: center; font-size: 48px; margin-top: 0;">Fire-Ready Forests Data Challenge</h1>

# Sprint 2 - Tasks 3-4

**Team Name:** 

**Members:**

- Member A
- Member B
- Member C

Use this notebook to report your solutions for tasks 3-4 of Sprint 2. Add code and markdown cells as needed to report your solutions.

### Task 3

Select a shapefile from a site other than Independence Lake. Using the `fastfuels-demo.ipynb` notebook as a base, generate the corresponding FastFuels sample and population treelists for the selected area and save them in a csv. Ensure that the output files include your `team-name` at the end of each filename.

In [1]:
import geopandas as gpd
from utils.treemap import TreeMapConnection
import matplotlib.pyplot as plt
import zipfile
import os
import pandas as pd
import shutil

In [2]:
sedg = gpd.read_file("./sedgwick_boundary.geojson")
sedg_utm = sedg.to_crs(5070) # We transform to UTM Coordinate Reference System (CRS).

In [4]:
zip_path = "./TreeMap2016_tree_table.zip"
extract_path = "./TreeMap2016" # Temporary directory
os.makedirs(extract_path, exist_ok=True)

# Extract the ZIP file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Delete the ZIP file after extraction
os.remove(zip_path)

csv_path = "./TreeMap2016/TreeMap2016_tree_table.csv"

# Convert CSV to Parquet
df = pd.read_csv(csv_path)
df.to_parquet("./TreeMap2016_tree_table.parquet", engine='pyarrow', index=False)

# Remove the CSV directory
shutil.rmtree(extract_path)

In [5]:
version="2016" 
seed=123

treemap_connection = TreeMapConnection(
    treemap_path=f"https://wifire-data.sdsc.edu/data/treemap/TreeMap{version}.tif", # This is the remote URL
    tree_table_path=f"./TreeMap{version}_tree_table.parquet",
    version=version,
)

treemap_raster_extraction = treemap_connection.extract_window(
    sedg_utm,
    projection_padding_meters=15 * treemap_connection.raster_resolution,
    interpolation_padding_cells=4,
)

treemap_plots = treemap_connection.get_plots_dataframe_from_raster(
    treemap_raster_extraction
)
tree_sample = treemap_connection.query_trees_by_plots(treemap_plots)

#uncomment to create the file
tree_sample.to_csv("sedgwick_sample_heatmappers.csv", index=False)

In [6]:
tree_population = tree_sample.expand_to_roi(
    "inhomogeneous_poisson",
    sedg_utm,
    plots=treemap_plots,
    intensity_resolution=15,
    seed=seed,
)

#uncomment to create the file
tree_population.to_csv("sedgwick_population_heatmappers.csv", index=False) 

In [ ]:
sample = pd.read_csv('sedgwick_sample_heatmappers.csv')
population = pd.read_csv('sedgwick_population_heatmappers.csv')

### Task 4

Compare the sample treelist with the population treelist. What do you think are the main reasons for any discrepancies between the two treelists?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

sample_df = pd.read_csv("sedgwick_sample_heatmappers.csv")
population_df = pd.read_csv("sedgwick_population_heatmappers.csv")

sample_count = len(sample_df)
population_count = len(population_df)
print(f"Sample Tree Count: {sample_count}")
print(f"Population Tree Count: {population_count}")

sample_avg_dia = sample_df["DIA"].mean()
population_avg_dia = population_df["DIA"].mean()
print(f"Average DIA (Sample): {sample_avg_dia:.2f}")
print(f"Average DIA (Population): {population_avg_dia:.2f}")

plt.hist(sample_df["DIA"], bins=20, alpha=0.5, label="Sample DIA")
plt.hist(population_df["DIA"], bins=20, alpha=0.5, label="Population DIA")
plt.xlabel("Diameter at Breast Height (DIA)")
plt.ylabel("Count")
plt.title("Comparison of Tree Diameters: Sample vs. Population")
plt.legend()
plt.show()

if "SPCD" in sample_df.columns:
    sample_species_counts = sample_df["SPCD"].value_counts()
    population_species_counts = population_df["SPCD"].value_counts()
    species_comparison_df = pd.concat(
        [sample_species_counts, population_species_counts],
        axis=1,
        keys=["Sample", "Population"]
    )
    print(species_comparison_df.head(10))


Some differences between the sample and population treelists typically occur because the population data infills any gaps that the sample misses. For exampole, smaller understory trees might be overlooked, and steep or dense forests make detection even harder. The sample usually focuses on measured plots, whereas the population tries to represent the entire area by modeling places where no direct measurements exist. Population models often assume standard distributions for characteristics like tree diameter and height, which can shift the overall size profile compared to the raw sample. Further on this, remote-sensing or allometric methods used for species identification may group similar trees together, affecting the result. Finally, we find that filling data gaps and interpolating plausible trees can naturally lead to differences in the number of trees and their distribution across the landscape.